# Full HIGGS Benchmark on Colab or Kaggle

**Author:** CosmicMagnetar | **Project:** Higgs Event Lab

This notebook runs the full Higgs benchmark on a large sample using the official UCI data link. It is designed for Colab or Kaggle and fetches data automatically.


## Environment bootstrap

This cell checks whether the project code is already present. If the notebook runs on a fresh Colab or Kaggle machine, it clones the repository once and installs the missing packages. Run this cell first so later cells can find the code.


In [ ]:
from pathlib import Path
import importlib.util
import os
import subprocess
import sys

REPO_URL = "https://github.com/CosmicMagnetar/higgs-event-lab.git"

root = Path(os.environ.get("HIGGS_REPO_ROOT", Path.cwd() / "higgs-event-lab"))
if not (root / "src" / "higgs_event_lab").exists():
    print("Cloning repository into", root)
    root.mkdir(parents=True, exist_ok=True)
    subprocess.run(
        ["git", "clone", "--depth", "1", REPO_URL, str(root)],
        check=True,
    )

print("Project root:", root)

for package in ("numpy", "pandas", "scikit-learn", "xgboost", "torch", "matplotlib"):
    if importlib.util.find_spec(package) is None:
        print(f"Installing {package}...")
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", package], check=False)

if str(root / "src") not in sys.path:
    sys.path.insert(0, str(root / "src"))

print("Environment ready.")


## Download the full HIGGS dataset

This cell fetches the original UCI HIGGS archive and extracts the compressed CSV. It skips the download when the file already exists. The archive is about 2.6 GB, so the first run takes time.


In [ ]:
import os
from pathlib import Path
import shutil
import sys
import urllib.request
import zipfile

root = Path(os.environ.get("HIGGS_REPO_ROOT", Path.cwd() / "higgs-event-lab"))
raw_path = root / "data" / "raw" / "HIGGS.csv.gz"
raw_path.parent.mkdir(parents=True, exist_ok=True)
url = "https://archive.ics.uci.edu/static/public/280/higgs.zip"

if not raw_path.exists():
    print("Downloading full HIGGS from UCI...")
    zip_path = raw_path.parent / "higgs_download.zip"
    urllib.request.urlretrieve(url, zip_path)
    with zipfile.ZipFile(zip_path) as archive:
        gz_name = [name for name in archive.namelist() if name.endswith(".csv.gz")][0]
        with archive.open(gz_name) as source, open(raw_path, "wb") as target:
            shutil.copyfileobj(source, target)
    zip_path.unlink()
print("Raw data ready:", raw_path)


## Prepare a one million event subset

This cell builds a larger processed sample for full-scale training. It uses the shared extraction helper and records a manifest. Adjust the row count if memory is limited.


In [ ]:
import os
import sys
from pathlib import Path

root = Path(os.environ.get("HIGGS_REPO_ROOT", Path.cwd() / "higgs-event-lab"))
if str(root / "src") not in sys.path:
    sys.path.insert(0, str(root / "src"))

from higgs_event_lab.data import extract_sample

raw_path = root / "data" / "raw" / "HIGGS.csv.gz"
out_path = root / "data" / "processed" / "higgs_sample_1m.csv.gz"

if not out_path.exists():
    extract_sample(
        source_path=raw_path,
        output_path=out_path,
        n_rows=1000000,
        compare_later_chunk=False,
    )
    print("Processed sample ready:", out_path)


## Research and authorship used in this run

Saumya Mishra owns the physics background, dataset guide, and QA plan that explain why the 21 low-level and 7 high-level features are the right problem here. The project lead tuned the models and prepared this Colab walkthrough so the numbers can be reproduced without talking about physics that the benchmark never claims. Use this note in the report and viva so the reader understands who wrote what.


## Train the tuned full benchmark

This cell launches the tuned run for logistic regression, XGBoost, and a wider compact MLP on the processed sample. It keeps the test split frozen across models and writes a single summary with per-model test predictions into the outputs folder next to the dashboard. Run it after the download and subset cells have finished.


In [ ]:
import os
import subprocess
import sys
from pathlib import Path

root = Path(os.environ.get("HIGGS_REPO_ROOT", Path.cwd() / "higgs-event-lab"))
script = root / "scripts" / "train_full_colab.py"

os.environ["HIGGS_SAMPLE_PATH"] = str(root / "data" / "processed" / "higgs_sample_1m.csv.gz")
result = subprocess.run([sys.executable, str(script)], cwd=root, check=False)
print("Tuned training exit code:", result.returncode)
